In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as img

import cv2
import itertools
import pathlib
import warnings
from PIL import Image
from random import randint
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef as MCC
from sklearn.metrics import balanced_accuracy_score as BAS
from sklearn.metrics import classification_report, confusion_matrix


from tensorflow import keras
from keras import layers
import tensorflow as tf
#import tensorflow_addons as tfa
from tensorflow.keras.preprocessing import image_dataset_from_directory
##from keras.utils.vis_utils import plot_model
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import Conv2D, Flatten
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator as IDG
from tensorflow.keras.layers import SeparableConv2D, BatchNormalization, GlobalAveragePooling2D

from distutils.dir_util import copy_tree, remove_tree

import os
#print(os.listdir("../input/alzheimer-mri-dataset/Dataset"))

print("TensorFlow Version:", tf.__version__)

In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

In [ ]:
class ReadDataset:
    def __init__(self, datasetpath, labels, image_shape):
        self.datasetpath = datasetpath
        self.labels = labels
        self.image_shape = image_shape
    def returListImages(self,):
        self.images = []
        for label in self.labels:
            self.images.append(list(pathlib.Path(os.path.join(self.datasetpath,
                                                              label)).glob('*.*')))
    def readImages(self,):
        self.returListImages()
        self.finalImages = []
        labels = []
        for label in range(len(self.labels)):
            for img in self.images[label]:
                img = cv2.imread(str(img))
                img = cv2.resize(img , self.image_shape)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img  = img/255
                self.finalImages.append(img)
                labels.append(label)
        images = np.array(self.finalImages)
        labels = np.array(labels)
        return images, labels

readDatasetObject = ReadDataset('sarscov2-ctscan-dataset',
                               ['COVID', 'non-COVID'],
                               (128,128))
images, labels = readDatasetObject.readImages()

images.shape, labels.shape

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)
X_train.shape,X_test.shape, y_train.shape,y_test.shape

In [ ]:
images_train = X_train
images_test = X_test

labels_train = y_train
labels_test = y_test

images_train.shape,images_test.shape, labels_train.shape,labels_test.shape

In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

In [ ]:
plt.figure(figsize = (12, 4))
for i in range(16):
    plt.subplot(2, 8, (i + 1))
    plt.imshow(images_train[i], cmap = 'gray')
    plt.title(labels_train[i])
plt.show()

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train_one_hot = to_categorical(labels_train)
y_train_one_hot.shape

In [ ]:
from tensorflow.keras.utils import to_categorical
y_test_one_hot = to_categorical(labels_test)
y_test_one_hot.shape

In [ ]:
y_test_one_hot.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

##SCA
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
import tensorflow as tf

class SpatialAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
        super(SpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class ChannelAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(units=input_shape[-1] // 2, activation='relu')
        self.dense2 = tf.keras.layers.Dense(units=input_shape[-1], activation='sigmoid')
        super(ChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class CombinedAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(CombinedAttentionNoiseLayer, self).__init__(**kwargs)
        self.spatial_attention = SpatialAttentionLayer()
        self.channel_attention = ChannelAttentionLayer()
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = self.spatial_attention(inputs)
        channel_attention_output = self.channel_attention(inputs)

        # Add spatial attention noise
        spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                    mean=0, stddev=self.spatial_noise_factor)

        # Add channel attention noise
        channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                    mean=0, stddev=self.channel_noise_factor)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)


In [ ]:
## more correct protection
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    x = Activation('relu')(x)
    return x

def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs = Input(shape=input_shape)
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs)

    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    # Global average pooling and fully connected layer
    x = GlobalAveragePooling2D()(x)
    ##x = Dense(128, activation='selu')(x)
    outputs = Dense(2, activation='sigmoid')(x)

    # Create the model
    model = Model(inputs, outputs)
    return model

# Instantiate the ResNet-18 model
resnet18 = build_resnet18()

# Display the model summary
#resnet18.summary()
resnet18.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
resnet18.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
resnet18.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
#resnet18.fit(images_train, y_train_one_hot
resnet18.evaluate(images_test, y_test_one_hot)

In [ ]:
images_test.shape

In [ ]:
'''random_indices = np.random.choice(497, 1, replace=False)
#adversarial_examples
X = images_test[random_indices]
adversarial_examples_X = adversarial_examples[random_indices]
y = y_test_one_hot[random_indices]

X.shape, y.shape, adversarial_examples_X.shape'''

In [ ]:
'''model1 = model
model = resnet18'''

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
#epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
epsilon_values = [0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(resnet18.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]
#epsilon_values = [0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=20,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(resnet18.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]
#epsilon_values = [0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(resnet18.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    
    x = Activation('relu')(x)
    return x

def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs = Input(shape=input_shape)
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    
    
    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    
    # Global average pooling and fully connected layer
    x = GlobalAveragePooling2D()(x)
    ##x = Dense(128, activation='selu')(x)
    outputs = Dense(2, activation='sigmoid')(x)

    # Create the model
    model = Model(inputs, outputs)
    return model

# Instantiate the ResNet-18 model
resnet18_clean = build_resnet18()

# Display the model summary
#resnet18.summary()
resnet18_clean.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
resnet18_clean.fit(images_train, y_train_one_hot, epochs=100, verbose=0, #callbacks = callbacks,
          validation_split=0.2)
resnet18_clean.fit(images_train, y_train_one_hot, epochs=100, verbose=0, #callbacks = callbacks,
          validation_split=0.2)

resnet18_clean.evaluate(images_test, y_test_one_hot)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
#epsilon_values = [0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18_clean
for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0.06]
#epsilon_values = [0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18_clean
for epsilon in epsilon_values:
    adversarial_examples_resnet = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples_resnet.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples_resnet = np.concatenate(adversarial_examples_resnet, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples_resnet, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

#adversarial_examples_resnet = adversarial_examples
# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

def plot_confusion_matrix(cm, classes, title='Confusion Matrix', cmap=plt.cm.Blues):
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

batch_size = 20

# Define epsilon values
epsilon_values = [0.06]
#epsilon_values = [0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(resnet18.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })
    
    class_names =[0, 1]
    # Generate confusion matrix
    cm = confusion_matrix(y_test_one_hot.argmax(axis=1), y_pred_binary.argmax(axis=1))
    plt.figure()
    plot_confusion_matrix(cm, classes=class_names, title=f'Confusion Matrix (Epsilon={epsilon})')
    plt.savefig(f'UMAN_sars_cov2_pgd_confusion_matrix_epsilon_{epsilon}.png', dpi=1024)
    plt.savefig(f'UMAN_sars_cov2_pgd_confusion_matrix_epsilon_{epsilon}.pdf', dpi=1024)
    
# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
resnet18.evaluate(adversarial_examples_resnet, y_test_one_hot)

In [ ]:
pred_adv = resnet18.predict(adversarial_examples)
pred_adv_resnet = resnet18_clean.predict(adversarial_examples_resnet)
pred_clean = resnet18.predict(images_test)
resnet18_clean_pred_clean = resnet18_clean.predict(images_test)

pred_adv.shape, pred_clean.shape, pred_adv_resnet.shape, resnet18_clean_pred_clean.shape

In [ ]:

y_test_pred_adv = np.argmax(pred_adv, axis=1)
y_test_pred_adv_resnet = np.argmax(pred_adv_resnet, axis=1)
y_test_pred_clean = np.argmax(pred_clean, axis=1)
y_test_resnet18_clean_pred_clean = np.argmax(resnet18_clean_pred_clean, axis=1)

y_test_pred_adv.shape, y_test_pred_adv_resnet.shape,y_test_pred_clean.shape, y_test_resnet18_clean_pred_clean.shape

In [ ]:
adversarial_examples_resnet.shape

In [ ]:
random_indices = np.random.choice(497, 1, replace=False)
#adversarial_examples
X = images_test[random_indices]
adversarial_examples_X = adversarial_examples[random_indices]
adversarial_examples_resnet_X = adversarial_examples_resnet[random_indices]

y_test_one_hot1 = np.argmax(y_test_one_hot, axis=1)
y = y_test_one_hot1[random_indices]

y1 = y_test_pred_adv[random_indices]
y2 = y_test_pred_clean[random_indices]

y3 = y_test_pred_adv_resnet[random_indices]
y4 = y_test_resnet18_clean_pred_clean[random_indices]

X.shape, y.shape, adversarial_examples_X.shape, adversarial_examples_resnet_X.shape, y1.shape, y2.shape, y3.shape, y4.shape

In [ ]:
adversarial_examples_resnet_X.shape

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(X)
grad_model = tf.keras.models.Model(
    resnet18.inputs, [resnet18.get_layer('trainable_combined_attention_layer_25').output, resnet18.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()


In [ ]:


# Load the original image
import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Convert heatmap_rescaled to integers for indexing
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Index jet_colors using heatmap_indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, X.shape[1:3])
# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Ensure the superimposed image has the same shape as the original image
#superimposed_img_array = superimposed_img_array.numpy()#.astype(np.uint8)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y2}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

# Save the plot as PNG with DPI 1024
plt.savefig('clean_uman.png', dpi=1024)
plt.savefig('clean_uman.pdf', dpi=1024)

plt.show()

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(adversarial_examples_X)
grad_model = tf.keras.models.Model(
    resnet18.inputs, [resnet18.get_layer('trainable_combined_attention_layer_25').output, resnet18.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()


In [ ]:

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Convert heatmap_rescaled to integers for indexing
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Index jet_colors using heatmap_indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, adversarial_examples_X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(adversarial_examples_X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Ensure the superimposed image has the same shape as the original image
#superimposed_img_array = superimposed_img_array.numpy()#.astype(np.uint8)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y1}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('pgd_denoised_uman.png', dpi=1024)
plt.savefig('pgd_denoised_uman.pdf', dpi=1024)

plt.show()

## resnet18 on adversarial examples and clean samples

In [ ]:
resnet18_clean

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(X)
grad_model = tf.keras.models.Model(
    resnet18_clean.inputs, [resnet18_clean.get_layer('activation_17').output, 
                            resnet18_clean.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()


In [ ]:

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Round the rescaled heatmap to integer values
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Index jet_colors using rounded heatmap indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y4}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('clean_resnet18.png', dpi=1024)
plt.savefig('clean_resnet18.pdf', dpi=1024)

plt.show()


In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(adversarial_examples_resnet_X)
grad_model = tf.keras.models.Model(
    resnet18_clean.inputs, [resnet18_clean.get_layer('activation_17').output, 
                            resnet18_clean.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()


In [ ]:

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Round the rescaled heatmap to integer values
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Index jet_colors using rounded heatmap indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, adversarial_examples_resnet_X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(adversarial_examples_resnet_X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y3}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('pgd_adv_resnet18.png', dpi=1024)
plt.savefig('pgd_adv_resnet18.pdf', dpi=1024)

plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 30

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]
#epsilon_values = [0]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18
for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(resnet18.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 30

# Define epsilon values
#epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]
epsilon_values = [0.06]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 30

# Define epsilon values
#epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]
epsilon_values = [0.06]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18_clean
for epsilon in epsilon_values:
    adversarial_examples_resnet = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=resnet18_clean,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples_resnet.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples_resnet = np.concatenate(adversarial_examples_resnet, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples_resnet, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
pred_adv = resnet18.predict(adversarial_examples)
pred_adv_resnet = resnet18_clean.predict(adversarial_examples_resnet)
pred_clean = resnet18.predict(images_test)
resnet18_clean_pred_clean = resnet18_clean.predict(images_test)

pred_adv.shape, pred_clean.shape, pred_adv_resnet.shape, resnet18_clean_pred_clean.shape
y_test_pred_adv = np.argmax(pred_adv, axis=1)
y_test_pred_adv_resnet = np.argmax(pred_adv_resnet, axis=1)
y_test_pred_clean = np.argmax(pred_clean, axis=1)
y_test_resnet18_clean_pred_clean = np.argmax(resnet18_clean_pred_clean, axis=1)

y_test_pred_adv.shape, y_test_pred_adv_resnet.shape,y_test_pred_clean.shape, y_test_resnet18_clean_pred_clean.shape

random_indices = np.random.choice(497, 1, replace=False)
#adversarial_examples
X = images_test[random_indices]
adversarial_examples_X = adversarial_examples[random_indices]
adversarial_examples_resnet_X = adversarial_examples_resnet[random_indices]

y_test_one_hot1 = np.argmax(y_test_one_hot, axis=1)
y = y_test_one_hot1[random_indices]

y1 = y_test_pred_adv[random_indices]
y2 = y_test_pred_clean[random_indices]

y3 = y_test_pred_adv_resnet[random_indices]
y4 = y_test_resnet18_clean_pred_clean[random_indices]

X.shape, y.shape, adversarial_examples_X.shape, adversarial_examples_resnet_X.shape, y1.shape, y2.shape, y3.shape, y4.shape

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(adversarial_examples_resnet_X)
grad_model = tf.keras.models.Model(
    resnet18_clean.inputs, [resnet18_clean.get_layer('activation_17').output, 
                            resnet18_clean.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Round the rescaled heatmap to integer values
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Index jet_colors using rounded heatmap indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, adversarial_examples_resnet_X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(adversarial_examples_resnet_X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y3}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('mim_adv_resnet18.png', dpi=1024)
plt.savefig('mim_adv_resnet18.pdf', dpi=1024)

plt.show()


In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(X)
grad_model = tf.keras.models.Model(
    resnet18_clean.inputs, [resnet18_clean.get_layer('conv2d_65').output, 
                            resnet18_clean.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Round the rescaled heatmap to integer values
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Index jet_colors using rounded heatmap indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y4}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('clean_resnet18_mim.png', dpi=1024)
plt.savefig('clean_resnet18_mim.pdf', dpi=1024)

plt.show()


In [ ]:

# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(X)
grad_model = tf.keras.models.Model(
    resnet18.inputs, [resnet18.get_layer('trainable_combined_attention_layer_25').output, resnet18.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

# Load the original image
import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Convert heatmap_rescaled to integers for indexing
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Index jet_colors using heatmap_indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, X.shape[1:3])
# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Ensure the superimposed image has the same shape as the original image
#superimposed_img_array = superimposed_img_array.numpy()#.astype(np.uint8)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y2}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

# Save the plot as PNG with DPI 1024
plt.savefig('clean_uman.png', dpi=1024)
plt.savefig('clean_uman.pdf', dpi=1024)

plt.show()

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(adversarial_examples_X)
grad_model = tf.keras.models.Model(
    resnet18.inputs, [resnet18.get_layer('trainable_combined_attention_layer_25').output, resnet18.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Convert heatmap_rescaled to integers for indexing
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Index jet_colors using heatmap_indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, adversarial_examples_X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(adversarial_examples_X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Ensure the superimposed image has the same shape as the original image
#superimposed_img_array = superimposed_img_array.numpy()#.astype(np.uint8)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y1}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('mim_denoised_uman.png', dpi=1024)
plt.savefig('mim_denoised_uman.pdf', dpi=1024)

plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18
for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18_clean
for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18
for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0.06]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3]

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18_clean
for epsilon in epsilon_values:
    adversarial_examples_resnet = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples_resnet.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples_resnet = np.concatenate(adversarial_examples_resnet, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples_resnet, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0.06]

# List to store results for each epsilon
results_per_epsilon = []
model = resnet18_clean
for epsilon in epsilon_values:
    adversarial_examples_resnet = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples_resnet.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples_resnet = np.concatenate(adversarial_examples_resnet, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples_resnet, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import os

# Specify the directory path
directory = '/working'

# Get a list of all files and directories in the directory
files_and_dirs = os.listdir(directory)

# Iterate over each item and delete only files
for item in files_and_dirs:
    item_path = os.path.join(directory, item)
    if os.path.isfile(item_path):
        os.remove(item_path)

print("All files in the directory have been deleted.")


In [ ]:
pred_adv = resnet18.predict(adversarial_examples)
pred_adv_resnet = resnet18_clean.predict(adversarial_examples_resnet)
pred_clean = resnet18.predict(images_test)
resnet18_clean_pred_clean = resnet18_clean.predict(images_test)

pred_adv.shape, pred_clean.shape, pred_adv_resnet.shape, resnet18_clean_pred_clean.shape
y_test_pred_adv = np.argmax(pred_adv, axis=1)
y_test_pred_adv_resnet = np.argmax(pred_adv_resnet, axis=1)
y_test_pred_clean = np.argmax(pred_clean, axis=1)
y_test_resnet18_clean_pred_clean = np.argmax(resnet18_clean_pred_clean, axis=1)

y_test_pred_adv.shape, y_test_pred_adv_resnet.shape,y_test_pred_clean.shape, y_test_resnet18_clean_pred_clean.shape

random_indices = np.random.choice(600, 1, replace=False)
#adversarial_examples
X = images_test[random_indices]
adversarial_examples_X = adversarial_examples[random_indices]
adversarial_examples_resnet_X = adversarial_examples_resnet[random_indices]

y_test_one_hot1 = np.argmax(y_test_one_hot, axis=1)
y = y_test_one_hot1[random_indices]

y1 = y_test_pred_adv[random_indices]
y2 = y_test_pred_clean[random_indices]

y3 = y_test_pred_adv_resnet[random_indices]
y4 = y_test_resnet18_clean_pred_clean[random_indices]

X.shape, y.shape, adversarial_examples_X.shape, adversarial_examples_resnet_X.shape, y1.shape, y2.shape, y3.shape, y4.shape

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(adversarial_examples_resnet_X)
grad_model = tf.keras.models.Model(
    resnet18_clean.inputs, [resnet18_clean.get_layer('conv2d_66').output, 
                            resnet18_clean.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Round the rescaled heatmap to integer values
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Index jet_colors using rounded heatmap indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, adversarial_examples_resnet_X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(adversarial_examples_resnet_X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y3}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('bim_adv_resnet18.png', dpi=1024)
plt.savefig('bim_adv_resnet18.pdf', dpi=1024)

plt.show()


In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(X)
grad_model = tf.keras.models.Model(
    resnet18_clean.inputs, [resnet18_clean.get_layer('conv2d_66').output, 
                            resnet18_clean.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Round the rescaled heatmap to integer values
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Index jet_colors using rounded heatmap indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y4}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('clean_resnet18_bim.png', dpi=1024)
plt.savefig('clean_resnet18_bim.pdf', dpi=1024)

plt.show()


In [ ]:

# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(X)
grad_model = tf.keras.models.Model(
    resnet18.inputs, [resnet18.get_layer('trainable_combined_attention_layer_25').output, resnet18.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

# Load the original image
import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Convert heatmap_rescaled to integers for indexing
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Index jet_colors using heatmap_indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, X.shape[1:3])
# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Ensure the superimposed image has the same shape as the original image
#superimposed_img_array = superimposed_img_array.numpy()#.astype(np.uint8)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y2}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

# Save the plot as PNG with DPI 1024
plt.savefig('clean_uman_bim.png', dpi=1024)
plt.savefig('clean_uman_bim.pdf', dpi=1024)

plt.show()

In [ ]:
# First, we create a model that maps the input image to the activations
# of the last conv layer as well as the output predictions
X_tensor = tf.convert_to_tensor(adversarial_examples_X)
grad_model = tf.keras.models.Model(
    resnet18.inputs, [resnet18.get_layer('trainable_combined_attention_layer_25').output, resnet18.output]
)

# Then, we compute the gradient of the top predicted class for our input image
# with respect to the activations of the last conv layer
with tf.GradientTape() as tape:
    last_conv_layer_output, preds = grad_model(X_tensor)
    class_channel = preds[:, round(np.mean(tf.argmax(preds[0]).numpy()))]

# This is the gradient of the output neuron (top predicted or chosen)
# with regard to the output feature map of the last conv layer
grads = tape.gradient(class_channel, last_conv_layer_output)

# This is a vector where each entry is the mean intensity of the gradient
# over a specific feature map channel
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# We multiply each channel in the feature map array
# by "how important this channel is" with regard to the top predicted class
# then sum all the channels to obtain the heatmap class activation
last_conv_layer_output = last_conv_layer_output[0]
heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# For visualization purpose, we will also normalize the heatmap between 0 & 1
heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
heatmap = heatmap.numpy()

import matplotlib.cm as cm

# Rescale heatmap to a range 0-255
heatmap_rescaled = 255 * heatmap

# Use jet colormap to colorize heatmap
jet = cm.get_cmap("viridis")

# Use RGB values of the colormap
jet_colors = jet(np.arange(256))[:, :3]

# Convert heatmap_rescaled to integers for indexing
heatmap_indices = np.round(heatmap_rescaled).astype(int)

# Index jet_colors using heatmap_indices
jet_heatmap = jet_colors[heatmap_indices]

# Resize the heatmap to match the original image size
heatmap_resized = tf.image.resize(jet_heatmap, adversarial_examples_X.shape[1:3])

# Convert the heatmap to an array
heatmap_array = tf.keras.preprocessing.image.img_to_array(heatmap_resized)

# Convert X to a tensor
X_tensor = tf.convert_to_tensor(adversarial_examples_X, dtype=tf.float32)

# Normalize X_tensor to have values between 0 and 1
X_normalized = X_tensor / 255.0

# Blend the heatmap with the normalized original image
alpha = 0.8  # Adjust the transparency level
superimposed_img_array = (alpha * heatmap_array) + ((1 - alpha) * X_tensor)

# Ensure the superimposed image has the same shape as the original image
#superimposed_img_array = superimposed_img_array.numpy()#.astype(np.uint8)

# Convert the array back to an image
superimposed_img = tf.keras.preprocessing.image.array_to_img(tf.squeeze(superimposed_img_array))

# Plot the images
fig = plt.figure(figsize=(12, 8))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(superimposed_img)
ax1.set_title(f'Prediction: {y1}')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(tf.squeeze(X))  # Squeeze the first dimension of X_tensor
ax2.set_title(f'Prediction: {y}')
ax2.axis('off')

plt.savefig('bim_denoised_uman.png', dpi=1024)
plt.savefig('bim_denoised_uman.pdf', dpi=1024)

plt.show()

In [ ]:
import tensorflow as tf

# Load the model
loaded_model = tf.keras.models.load_model("your_model_name.tf")

# Now you can use the loaded model for inference or further training

In [ ]:
import tensorflow as tf
loaded_model.save("resnet_train_sca_covid_ct1.h5")

In [ ]:
loaded_model1 = tf.keras.models.load_model("resnet_train_sca_covid_ct1.h5", 
                                          custom_objects = {'TrainableCombinedAttentionLayer': TrainableCombinedAttentionLayer,
                                                            'CombinedAttentionNoiseLayer': CombinedAttentionNoiseLayer})

In [ ]:
loaded_model1.evaluate(images_test, y_test_one_hot)

In [ ]:
loaded_model.evaluate(images_test, y_test_one_hot)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    
    x = Activation('relu')(x)
    return x

def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs = Input(shape=input_shape)
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    
    
    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    
    # Global average pooling and fully connected layer
    x = GlobalAveragePooling2D()(x)
    ##x = Dense(128, activation='selu')(x)
    outputs = Dense(2, activation='sigmoid')(x)

    # Create the model
    model = Model(inputs, outputs)
    return model

# Instantiate the ResNet-18 model
resnet18_clean = build_resnet18()

# Display the model summary
#resnet18.summary()
resnet18_clean.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
resnet18_clean.fit(images_train, y_train_one_hot, epochs=100, verbose=0, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
resnet18_clean.fit(images_train, y_train_one_hot, epochs=100, verbose=0, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
resnet18_clean.evaluate(images_test, y_test_one_hot)